In [2]:
!pip install git+https://github.com/huggingface/transformers
# !pip install librosa
!pip install evaluate>=0.30
!pip install jiwer
# !pip install gradio
!pip install -q bitsandbytes datasets accelerate
!pip install git+https://github.com/huggingface/peft.git@main

  Cloning https://github.com/huggingface/transformers to c:\users\alenb\appdata\local\temp\pip-req-build-ck74k9bq
  Resolved https://github.com/huggingface/transformers to commit 342961f6696115970593a8ec8e727145cb5a499f
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for transformers: filename=transformers-4.52.0.dev0-py3-none-any.whl size=11296129 sha256=1c0c102ce9124477b1848f9e0c33d660d6a86f415979728e81b0972ef311b412
  Stored in directory: C:\Users\alenb\AppData\Local\Temp\pip-ephem-wheel-cache-plfykkue\wheels\c0\14\d6\6c9a5582d2ac191ec0a483be151a4495fe1eb2a6706ca49f1b
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.3
  

  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers 'C:\Users\alenb\AppData\Local\Temp\pip-req-build-ck74k9bq'


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 5.1 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/peft.git (to revision main) to c:\users\alenb\appdata\local\temp\pip-req-build-l4gdw4kg
  Resolved https://github.com/huggingface/peft.git to commit 8af29c646860e617b641225caf7ef47f7c3dcd26
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/peft.git 'C:\Users\alenb\AppData\Local\Temp\pip-req-build-l4gdw4kg'


In [ ]:
from modules.core import llm_chain

llm_chain.start_ollama_server()

print("Gõ 'exit' hoặc 'quit' để thoát.")
print("Để gửi ảnh: image <đường_dẫn_ảnh> [nội dung_tin_nhắn]")

while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in ["exit", "quit"]:
        print("🛑 Kết thúc chat.")
        break

    image_path = None
    text = user_input

    if user_input.lower().startswith("image "):
        parts = user_input.split(" ", 2)
        if len(parts) > 1:
            image_path = parts[1]
            text = parts[2] if len(parts) > 2 else ""
        else:
            print("⚠️ Cú pháp sai. Dùng: image <đường_dẫn_ảnh> [tin nhắn]")
            continue

    bot_response = llm_chain.chat(session_id=12, message=text, image_path=image_path)
    print(f"\nAssistant: {bot_response}")

In [ ]:
import os
import modules.config as config
os.path.exists(config.MEMORY_CHAT_PATH)

In [ ]:
import ollama

messages = [{"role": "system", "content": """Bạn là trợ lý AI /no_think"""},
            {"role": "user", "content": """1+1=?, so sánh 11.91 và 11.19 """}]
stream = ollama.chat(model="qwen3:8b", messages=messages,
                        stream=True)

In [ ]:
for stre in stream:
    print(stre.message.content, end="")

In [8]:
import whisperx
model = whisperx.load_model("weights/speech2text", "cpu", compute_type="int8")


No language specified, language will be first be detected for each audio file (increases inference time).


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.1.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint d:\Program\conda\envs\Hana\lib\site-packages\whisperx\assets\pytorch_model.bin`


Model was trained with pyannote.audio 0.0.1, yours is 3.3.2. Bad things might happen unless you revert pyannote.audio to 0.x.
Model was trained with torch 1.10.0+cu102, yours is 2.7.0+cpu. Bad things might happen unless you revert torch to 1.x.


In [9]:
audio = whisperx.load_audio(r"output_recording.wav")
segments = model.transcribe(audio, batch_size=8, language="vi")
text = " ".join([segment["text"] for segment in segments["segments"]])

In [10]:
text

'hoa hoa hoa hoa hoa hoa hoa xin chào à lô à lô à lô kết thúc kìa sao nó chọng mới kích lên ô kia là phía đấy phía đấy'

In [12]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "suzii/vi-whisper-large-v3-turbo-v1"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

Device set to use cpu


In [13]:
result = pipe("output_recording.wav", return_timestamps=True)
result

d:\Program\conda\envs\Hana\lib\site-packages\transformers\models\whisper\generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


{'text': ' tính xin chào alo alo alo kết thúc ghi âm sao nó chậm mới cất dần ok là phế vậy ngon đều có thể nói',
 'chunks': [{'timestamp': (0.96, 0.0),
   'text': ' tính xin chào alo alo alo kết thúc ghi âm'},
  {'timestamp': (0.0, None),
   'text': ' sao nó chậm mới cất dần ok là phế vậy ngon đều có thể nói'}]}

In [2]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, WhisperProcessor
from transformers import pipeline
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
peft_config = PeftConfig.from_pretrained("linl03/viv3-large-lora-addapter")

# 2. Tải model Whisper gốc
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    peft_config.base_model_name_or_path,
    torch_dtype=torch.float16,
    device_map="cuda"
)

# 3. Gắn adapter LoRA vào model
model = PeftModel.from_pretrained(model, "linl03/viv3-large-lora-addapter")
processor = WhisperProcessor.from_pretrained(peft_config.base_model_name_or_path, language="vi", task="transcribe")


ModuleNotFoundError: Could not import module 'BloomPreTrainedModel'. Are this object's requirements defined correctly?

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device="cuda",
)

# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
# sample = dataset[0]["audio"]
result = pipe(r"C:\Users\alenb\Desktop\Test_case_2.WAV", return_timestamps=True)
print(result["text"])

Device set to use cpu
d:\Program\conda\envs\Hana\lib\site-packages\transformers\models\whisper\generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
